# Capítulo 4 · Contratos de alcance

El agente del capítulo 2, con sus reglas comprobadas al editar y al terminar, recibe ahora el contrato de cada tarea: objetivo, rutas que puede cambiar, rutas prohibidas y criterios de aceptación. Se comparan tres variantes: sin contrato, con el contrato en la petición y con el contrato comprobado al editar y en la puerta.

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar el modelo, el repositorio y las herramientas

In [ ]:
!pip install -q transformers==5.19.0  # las versiones con las que se comprobó el libro

In [ ]:
"""Capítulo 4: contratos de alcance."""
import importlib
import io
import json
import re
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen3-4B-Instruct-2507"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-1.7B"              # sin GPU: más pequeño, y peor agente
RAIZ = Path.cwd()                           # donde están datos/ y comun_libro/
REPOSITORIO = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
               "archive/refs/heads/main.zip")
TAREAS = ["T01", "T02", "T03", "T04", "T05", "T10", "T11", "T13", "T14", "T15",
          "T17", "T18", "T19", "T20"]       # las tareas de una sola sesión
MAX_PASOS = 30             # llamadas al modelo por tarea
MAX_TOKENS_PASO = 600      # tokens nuevos por llamada
LIMITE_ENTRADA = 12_000    # tokens de la conversación: más no cabe en la T4
RESULTADOS_INTACTOS = 2    # los últimos resultados de herramientas no se recortan
MAX_RECHAZOS = 3           # veces que la puerta devuelve el trabajo
hr = None                  # el módulo de herramientas, tras preparar()



def cargar(nombre=None):
    """Carga el modelo y su tokenizador: en la GPU, en float16; si no, en la CPU."""
    gpu = torch.cuda.is_available()
    if nombre is None:
        nombre = MODELO_GPU if gpu else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre, dtype=torch.float16 if gpu else torch.float32)
    modelo.to("cuda" if gpu else "cpu").eval()
    print(f"Modelo {nombre} en {'cuda' if gpu else 'cpu'}")
    return tok, modelo


def preparar():
    """El repositorio de Ciclos Norte, sus tareas y las herramientas del agente.
    Si no están (en Colab), se descargan del repositorio del libro."""
    global hr
    if not (RAIZ / "comun_libro" / "herramientas_repo.py").exists():
        with urllib.request.urlopen(REPOSITORIO) as respuesta:
            archivo = zipfile.ZipFile(io.BytesIO(respuesta.read()))
        for nombre in archivo.namelist():
            relativa = nombre.split("/", 1)[1]
            if relativa.startswith(("datos/", "comun_libro/")) and not nombre.endswith("/"):
                destino = RAIZ / relativa
                destino.parent.mkdir(parents=True, exist_ok=True)
                destino.write_bytes(archivo.read(nombre))
    sys.path.insert(0, str(RAIZ / "comun_libro"))
    hr = importlib.import_module("herramientas_repo")
    print(f"Repositorio en {hr.REPO}: {len(hr.archivos_de(hr.REPO))} archivos")

In [ ]:
tok, modelo = cargar()
preparar()

## 2. El modelo

In [ ]:
def plantilla(tok, mensajes):
    """La conversación como texto, con las herramientas, en la plantilla del modelo."""
    return tok.apply_chat_template(mensajes, tools=hr.ESQUEMAS, tokenize=False,
                                   add_generation_prompt=True)


def generar(tok, modelo, texto):
    """Una respuesta del modelo, con decodificación voraz. Devuelve el texto y los
    tokens de entrada y de salida."""
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=MAX_TOKENS_PASO,
                                 do_sample=False, temperature=None, top_p=None, top_k=None,
                                 prefill_chunk_size=1024, pad_token_id=tok.eos_token_id)
    n_entrada = entrada["input_ids"].shape[1]
    nuevos = salida[0, n_entrada:].cpu()
    del entrada, salida
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # cada paso tiene otra longitud: sin esto la
        #                           memoria reservada crece hasta llenar la GPU
    return tok.decode(nuevos, skip_special_tokens=True).strip(), n_entrada, len(nuevos)


def leer_llamadas(respuesta):
    """Las llamadas {name, arguments} que el modelo escribe entre <tool_call> y
    </tool_call>. Una llamada que no se entiende queda con name None."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)(?:</tool_call>|$)", respuesta, re.S):
        try:
            datos = json.loads(bloque.strip(), strict=False)
        except ValueError:
            datos = None
        if isinstance(datos, dict) and "name" in datos:
            argumentos = datos.get("arguments", {})
            llamadas.append({"name": datos["name"],
                             "arguments": argumentos if isinstance(argumentos, dict)
                             else {}})
        else:
            llamadas.append({"name": None, "arguments": {}})
    return llamadas


def recortar_conversacion(tok, mensajes):
    """Si la conversación pasa de LIMITE_ENTRADA tokens, cambia los resultados de
    herramientas más antiguos por un aviso corto. Devuelve el texto para el modelo y
    cuántos ha recortado (None si ni así cabe)."""
    texto = plantilla(tok, mensajes)
    recortados = 0
    candidatos = [m for m in mensajes if m["role"] == "tool"
                  and not m.get("recortado")][:-RESULTADOS_INTACTOS or None]
    while len(tok(texto)["input_ids"]) > LIMITE_ENTRADA:
        if not candidatos:
            return texto, None
        mensaje = candidatos.pop(0)
        mensaje["content"] = json.dumps(
            {"aviso": "Resultado antiguo recortado para ahorrar memoria; vuelve a "
                      "pedirlo si lo necesitas.",
             "inicio": mensaje["content"][:150]}, ensure_ascii=False)
        mensaje["recortado"] = True
        recortados += 1
        texto = plantilla(tok, mensajes)
    return texto, recortados

## 3. Las reglas del capítulo 2

`comprobar_reglas` tiene un parámetro nuevo, `exigir_prueba`, para cuando el contrato prohíbe tocar `tests/`.

In [ ]:
SISTEMA = (
    "Eres un agente de programación que trabaja en el repositorio de Ciclos Norte, "
    "una tienda de bicicletas de Oviedo. Es un programa de gestión en Python con "
    "varios módulos (pedidos, devoluciones, existencias, facturas, avisos…), sus "
    "pruebas en la carpeta tests/ y unos datos de ejemplo. Recibirás una petición de "
    "alguien de la tienda y tienes que resolverla cambiando el código con las "
    "herramientas: listar archivos, leer por rangos de líneas, buscar, editar por "
    "sustitución exacta, ejecutar las pruebas y unas pocas órdenes permitidas. "
    "Trabaja paso a paso, con una herramienta cada vez. Cuando hayas terminado, "
    "responde sin llamar a ninguna herramienta, empezando por «TERMINADO:» y "
    "explicando en una o dos frases qué has cambiado. Si no puedes terminar, empieza "
    "por «NO TERMINADO:» y explica por qué.")


AGENTS_REGLAS = """# AGENTS.md · Ciclos Norte

Reglas del repositorio. Cada una se comprueba al final de tu trabajo.

R1. Antes de editar un archivo, léelo con leer_archivo.
R2. No cambies nada de datos/.
R3. Cambia docs/ solo si la petición pide cambiar las condiciones generales.
R4. No borres ni cambies pruebas que ya existen. Excepción: si la petición cambia una
    regla del negocio, actualiza las pruebas de la regla antigua y dilo en tu respuesta.
R5. Añade tus pruebas en el archivo del módulo (tests/test_envios.py para envios.py):
    sustituye la línea `if __name__ == "__main__":` por una clase nueva con tu prueba,
    seguida de esa misma línea. Importa dentro de la prueba lo que necesites.
R6. No crees módulos nuevos fuera de tests/.
R7. «TERMINADO» exige: cambios en el código del programa, una comprobación nueva
    en tests/ y ejecutar_pruebas en verde después de tu último cambio.
R8. Tu respuesta final nombra todos los archivos que has cambiado.
"""


PIDE_DOCS = re.compile(r"(actualiz\w*|y en las)\s[^.]{0,40}condiciones\s+generales", re.I)


def lineas_del_diff(diff):
    """Las líneas añadidas y quitadas de cada archivo de un diff unificado:
    {ruta: {"+": [...], "-": [...]}}."""
    archivos, actual = {}, None
    for linea in diff.splitlines():
        if linea.startswith("+++ b/"):
            actual = archivos.setdefault(linea[6:], {"+": [], "-": []})
        elif linea.startswith("--- a/") or actual is None:
            continue
        elif linea[:1] in "+-" and linea[1:].strip():
            actual[linea[0]].append(linea[1:])
    return archivos


def comprobar_reglas(tarea, registro, cambios, respuesta, exigir_prueba=True,
                     permite_docs=None):
    """Las reglas incumplidas, como {regla: evidencia}. Mira lo que de verdad ha
    cambiado (el diff), el registro de llamadas y la respuesta final. Con
    «exigir_prueba» en False, la R7 no pide una comprobación nueva; «permite_docs»,
    si se da, decide la R3 en lugar de la petición."""
    if permite_docs is None:
        permite_docs = bool(PIDE_DOCS.search(tarea["peticion"]))
    rotas = {}
    lineas = lineas_del_diff(cambios["diff"])
    leidos = set()
    for r in registro:
        ruta = r["argumentos"].get("ruta", "")
        if r["herramienta"] == "leer_archivo" and r["ok"]:
            leidos.add(ruta)
        elif r["herramienta"] == "editar" and r["ok"] and ruta not in leidos:
            rotas.setdefault("R1", f"editaste {ruta} sin leerlo antes")
    for ruta in cambios["cambiados"]:
        if ruta.startswith("datos/"):
            rotas["R2"] = f"cambiaste {ruta}"
        if ruta.startswith("docs/") and not permite_docs:
            rotas["R3"] = f"cambiaste {ruta} y la petición no lo pide"
        if ruta.startswith("tests/") and ruta in cambios["nuevos"]:
            rotas["R5"] = f"creaste {ruta}; las pruebas van en el archivo del módulo"
        if not ruta.startswith(("tests/", "docs/", "datos/")) and ruta in cambios["nuevos"]:
            rotas["R6"] = f"creaste el módulo {ruta}"
    quitadas = [r for r, l in lineas.items() if r.startswith("tests/") and l["-"]]
    if quitadas and not re.search(r"prueba", respuesta or "", re.I):
        rotas["R4"] = (f"cambiaste pruebas que ya existían en {', '.join(quitadas)} "
                       "y tu respuesta no lo dice")
    if respuesta and dice_terminado(respuesta):
        programa = [r for r in cambios["cambiados"]
                    if not r.startswith(("tests/", "docs/", "datos/"))]
        prueba_nueva = any(re.search(r"def test_|assert", l)
                           for r, l2 in lineas.items() if r.startswith("tests/")
                           for l in l2["+"])
        ediciones = [i for i, r in enumerate(registro)
                     if r["herramienta"] in ("editar", "crear_archivo") and r["ok"]]
        despues = registro[ediciones[-1] + 1:] if ediciones else registro
        pruebas = [r for r in despues if r["herramienta"] == "ejecutar_pruebas"]
        falta = [texto for cumple, texto in (
            (programa, "no hay cambios en el código del programa"),
            (prueba_nueva or not exigir_prueba,
             "no hay ninguna comprobación nueva en tests/"),
            (pruebas and pruebas[-1]["resultado"].get("todas_pasan"),
             "no hay una ejecución de las pruebas en verde después del último cambio"))
            if not cumple]
        if falta:
            rotas["R7"] = "; ".join(falta)
        sin_nombrar = [r for r in cambios["cambiados"]
                       if Path(r).name not in respuesta]
        if sin_nombrar:
            rotas["R8"] = f"tu respuesta no nombra {', '.join(sin_nombrar)}"
    return rotas


def dice_terminado(respuesta):
    """True si la respuesta final dice que la tarea está hecha. Una respuesta vacía
    no dice nada."""
    if not respuesta.strip():
        return False
    if re.search(r"NO\s+TERMINADO", respuesta, re.I):
        return False
    if re.search(r"TERMINADO", respuesta):
        return True
    return not re.search(r"no (he podido|puedo|consigo|es posible)|necesito|\?",
                         respuesta, re.I)


class HerramientasConReglas:
    """Las herramientas del agente con las reglas de proceso comprobadas antes de
    cada edición: si la edición incumple R1, R2, R3, R5 o R6, no se hace y el agente
    recibe la regla. El resto de llamadas pasan tal cual."""

    def __init__(self, herramientas, tarea):
        self.herramientas = herramientas
        self.registro = herramientas.registro
        self.tarea = tarea
        self.leidos = set()

    def regla_rota(self, nombre, ruta):
        """La regla que incumpliría editar o crear «ruta», o None."""
        existe = self.herramientas.espacio.ruta(ruta).exists()
        if ruta.startswith("datos/"):
            return "R2: no cambies nada de datos/."
        if ruta.startswith("docs/") and not self.permite_docs():
            return "R3: la petición no pide cambiar las condiciones generales."
        if nombre == "crear_archivo" and ruta.startswith("tests/"):
            return ("R5: no crees archivos de pruebas; añade tu prueba al archivo del "
                    "módulo, sustituyendo la línea if __name__ == \"__main__\":")
        if nombre == "crear_archivo" and not existe:
            return "R6: no crees módulos nuevos fuera de tests/."
        if nombre == "editar" and existe and ruta not in self.leidos:
            return f"R1: lee {ruta} con leer_archivo antes de editarlo."
        return None

    def permite_docs(self):
        """True si la petición pide cambiar las condiciones generales (R3)."""
        return bool(PIDE_DOCS.search(self.tarea["peticion"]))

    def llamar(self, nombre, argumentos):
        """Como Herramientas.llamar, pero comprobando las reglas antes de editar."""
        ruta = str((argumentos or {}).get("ruta", ""))
        if nombre in ("editar", "crear_archivo"):
            regla = self.regla_rota(nombre, ruta)
            if regla:
                resultado = {"error": "Edición rechazada por AGENTS.md. " + regla}
                self.registro.append({"herramienta": nombre, "argumentos": argumentos,
                                      "ok": False, "segundos": 0,
                                      "resultado": resultado})
                return resultado
        resultado = self.herramientas.llamar(nombre, argumentos)
        if nombre == "leer_archivo" and "error" not in resultado:
            self.leidos.add(ruta)
        return resultado

## 4. El contrato

El contrato de una tarea, tal como lo ve el agente, y las comprobaciones, que se pueden probar sin modelo.

In [ ]:
def contrato_texto(alcance):
    """Un contrato (como datos/tareas/Txx/alcance.json) escrito para el agente:
    objetivo, rutas que puede cambiar, rutas prohibidas y criterios de aceptación
    numerados (C1, C2…)."""
    criterios = "\n".join(f"C{i}. {c}" for i, c in
                          enumerate(alcance["criterios_aceptacion"], 1))
    return ("Contrato de la tarea\n"
            f"Objetivo: {alcance['objetivo']}\n"
            f"Puedes cambiar: {', '.join(alcance['rutas_permitidas'])}\n"
            f"No toques: {', '.join(alcance['rutas_prohibidas'])}\n"
            f"Criterios de aceptación:\n{criterios}\n"
            "El contrato manda sobre AGENTS.md: si prohíbe tests/, no añadas pruebas.")


def prohibe_pruebas(alcance):
    """True si el contrato prohíbe cambiar tests/ (por ejemplo, una refactorización
    que debe pasar las pruebas sin tocarlas)."""
    return hr.coincide("tests/x.py", alcance["rutas_prohibidas"])


def comprobar_contrato(alcance, cambios, respuesta):
    """Lo que incumple el contrato al terminar: archivos fuera de las rutas permitidas
    o en las prohibidas, y criterios que la respuesta no explica cómo se cumplen."""
    rotas = {}
    rutas = hr.comprobar_alcance(cambios["cambiados"], alcance)
    fuera = sorted(set(rutas["fuera_de_alcance"]) | set(rutas["prohibidos"]))
    if fuera:
        rotas["Alcance"] = f"cambiaste {', '.join(fuera)}, que el contrato no permite"
    numeros = range(1, len(alcance["criterios_aceptacion"]) + 1)
    sin_explicar = [f"C{i}" for i in numeros
                    if not re.search(rf"\bC{i}\b", respuesta or "")]
    if sin_explicar:
        rotas["Criterios"] = ("tu respuesta no dice cómo se cumple "
                              + ", ".join(sin_explicar)
                              + " (cita cada criterio por su número)")
    return rotas


class HerramientasConContrato(HerramientasConReglas):
    """Las herramientas con las reglas del capítulo 2 y, además, el contrato: no se
    puede editar ni crear nada fuera de las rutas permitidas."""

    def __init__(self, herramientas, tarea, alcance):
        super().__init__(herramientas, tarea)
        self.alcance = alcance

    def permite_docs(self):
        """El contrato manda sobre la R3: docs/ se puede cambiar si lo permite."""
        return hr.coincide("docs/condiciones_generales.md", self.alcance["rutas_permitidas"])

    def regla_rota(self, nombre, ruta):
        """La regla o la parte del contrato que incumpliría la edición, o None."""
        alcance = self.alcance
        if hr.coincide(ruta, alcance["rutas_prohibidas"]):
            return f"Contrato: {ruta} está entre las rutas que no puedes tocar."
        if not hr.coincide(ruta, alcance["rutas_permitidas"]):
            return (f"Contrato: {ruta} no está entre las rutas que puedes cambiar ("
                    + ", ".join(alcance["rutas_permitidas"]) + ").")
        return super().regla_rota(nombre, ruta)

In [ ]:
tarea = hr.leer_tarea("T17")
print(contrato_texto(tarea["alcance"]))
with hr.Espacio() as espacio:
    h = HerramientasConContrato(hr.Herramientas(espacio), tarea, tarea["alcance"])
    print(h.llamar("editar", {"ruta": "envios.py", "buscar": "a", "poner": "b"}))
    print(comprobar_contrato(tarea["alcance"], hr.diferencias(espacio.origen, espacio.carpeta),
                             "TERMINADO: se cumplen C1 y C2."))

## 5. El agente

La tarea del nuevo precio de la revisión, sin contrato y con el contrato comprobado. Fíjate en el primer archivo que lee el agente.

In [ ]:
def agente(tok, modelo, tarea, contrato=None, alcance=None, max_pasos=MAX_PASOS,
           ver=False):
    """El agente del capítulo 2 (reglas comprobadas al editar y al terminar) con un
    contrato: sin él (None), «visible» en la petición o «comprobado», visible y además
    vigilado al editar y en la puerta. El contrato es el de la tarea, salvo que se
    pase otro en «alcance». Devuelve la fila."""
    inicio = time.time()
    alcance = alcance or tarea["alcance"]
    peticion = tarea["peticion"]
    if contrato:
        peticion += "\n\n" + contrato_texto(alcance)
    mensajes = [{"role": "system", "content": SISTEMA + "\n\n" + AGENTS_REGLAS},
                {"role": "user", "content": peticion}]
    final, pasos, entrada, salida, rechazos = None, 0, 0, 0, 0
    with hr.Espacio() as espacio:
        base = hr.Herramientas(espacio)
        if contrato == "comprobado":
            herramientas = HerramientasConContrato(base, tarea, alcance)
        else:
            herramientas = HerramientasConReglas(base, tarea)
        for pasos in range(1, max_pasos + 1):
            texto, recortados = recortar_conversacion(tok, mensajes)
            if recortados is None:
                break
            respuesta, n_entrada, n_salida = generar(tok, modelo, texto)
            entrada, salida = entrada + n_entrada, salida + n_salida
            llamadas = leer_llamadas(respuesta)
            if not llamadas:
                mensajes.append({"role": "assistant", "content": respuesta})
                if ver:
                    print(f"[{pasos}] {respuesta}")
                if dice_terminado(respuesta) and rechazos < MAX_RECHAZOS:
                    cambios = hr.diferencias(espacio.origen, espacio.carpeta)
                    rotas = comprobar_reglas(
                        tarea, base.registro, cambios, respuesta,
                        exigir_prueba=not (contrato and prohibe_pruebas(alcance)),
                        permite_docs=(herramientas.permite_docs()
                                      if contrato == "comprobado" else None))
                    if contrato == "comprobado":
                        rotas.update(comprobar_contrato(alcance, cambios, respuesta))
                    if rotas:
                        rechazos += 1
                        aviso = ("Todavía no está terminado: "
                                 + "; ".join(f"{r}: {t}" for r, t in sorted(rotas.items()))
                                 + ". Sigue trabajando.")
                        mensajes.append({"role": "user", "content": aviso})
                        if ver:
                            print(f"    PUERTA: {aviso}")
                        continue
                final = respuesta
                break
            pensamiento = respuesta.split("<tool_call>")[0].strip()
            mensajes.append({"role": "assistant", "content": pensamiento,
                             "tool_calls": [{"type": "function", "function": l}
                                            for l in llamadas if l["name"]]})
            for llamada in llamadas:
                if llamada["name"] is None:
                    resultado = {"error": "No se entiende la llamada a la herramienta."}
                else:
                    resultado = herramientas.llamar(llamada["name"], llamada["arguments"])
                if ver:
                    print(f"[{pasos}] {llamada['name']}"
                          f"({json.dumps(llamada['arguments'], ensure_ascii=False)[:90]})"
                          f" -> {json.dumps(resultado, ensure_ascii=False)[:110]}")
                mensajes.append({"role": "tool", "name": llamada["name"] or "?",
                                 "content": json.dumps(resultado, ensure_ascii=False)})
        evaluacion = hr.evaluar_intento(espacio, tarea)
    terminado = final is not None and dice_terminado(final)
    return {"tarea": tarea["id"], "tipo": tarea["meta"]["tipo"],
            "resuelta": evaluacion["resuelta"], "dice_terminado": terminado,
            "terminado_falso": terminado and not evaluacion["resuelta"],
            "fuera_de_alcance": evaluacion["fuera_de_alcance"],
            "prohibidos": evaluacion["prohibidos"], "rechazos": rechazos,
            "ocultas": evaluacion["ocultas"], "visibles": evaluacion["visibles"],
            "pasos": pasos, "tokens_entrada": entrada, "tokens_salida": salida,
            "segundos": round(time.time() - inicio, 1),
            "respuesta": final or "(sin respuesta final)",
            "llamadas": [[r["herramienta"], r["argumentos"].get("ruta", ""), r["ok"]]
                         for r in base.registro],
            "diff": evaluacion["diff"]}

In [ ]:
for nombre in ("sin contrato", "contrato comprobado"):
    print(f"--- {nombre}")
    fila = agente(tok, modelo, hr.leer_tarea("T17"), ver=True, **VARIANTES[nombre])
    print("Resuelta:", fila["resuelta"], "· Fuera de alcance:", fila["fuera_de_alcance"])

## 6. Las catorce tareas con las tres variantes

Tardan unas tres horas en una GPU de escritorio. Para una primera prueba, pasa `tareas=TAREAS[:4]` a `probar`.

In [ ]:
VARIANTES = {
    "sin contrato": {},
    "contrato visible": {"contrato": "visible"},
    "contrato comprobado": {"contrato": "comprobado"},
}


def resumen(filas):
    """Los totales de una variante."""
    return {"tareas": len(filas),
            "resueltas": sum(f["resuelta"] for f in filas),
            "dice_terminado": sum(f["dice_terminado"] for f in filas),
            "terminado_falso": sum(f["terminado_falso"] for f in filas),
            "con_fuera_de_alcance": sum(bool(f["fuera_de_alcance"]) for f in filas),
            "archivos_fuera": sum(len(f["fuera_de_alcance"]) for f in filas),
            "ocultas": sum(f["ocultas"] for f in filas),
            "pasos": sum(f["pasos"] for f in filas),
            "tokens_entrada": sum(f["tokens_entrada"] for f in filas),
            "segundos": round(sum(f["segundos"] for f in filas), 1)}


def probar(tok, modelo, opciones, tareas=TAREAS):
    """Una variante con todas las tareas: sus filas y su resumen."""
    filas = []
    for identificador in tareas:
        fila = agente(tok, modelo, hr.leer_tarea(identificador), **opciones)
        filas.append(fila)
        print(f"  {identificador} {'resuelta' if fila['resuelta'] else 'no      '}"
              f"  {'dice terminado' if fila['dice_terminado'] else 'no lo dice    '}"
              f"  fuera={len(fila['fuera_de_alcance'])}"
              f"  pasos={fila['pasos']:2d}  {fila['segundos']:6.1f} s", flush=True)
    return {"filas": filas, "resumen": resumen(filas)}

In [ ]:
resultados = {}
for nombre, opciones in VARIANTES.items():
    print(f"== {nombre}")
    resultados[nombre] = probar(tok, modelo, opciones)
    print(resultados[nombre]["resumen"])

## Ejercicios

Los archivos de resultados que citan los ejercicios están en `datos/resultados/capitulo_4/`, que `preparar` descarga con los datos.

1. Escribe el contrato de la tarea de los agotados (T05) sin mirar el suyo, solo con la petición. Compáralo después con `datos/tareas/T05/alcance.json`. ¿Qué se te ha pasado? ¿Qué has puesto de más? *(Sin GPU.)*

2. En la tarea del nuevo precio, el contrato pone `config.py` el primero de las rutas permitidas. Cambia el orden y ponlo el último. ¿Cambia el primer archivo que lee el agente? ¿Y el resultado? *(GPU, unos 15 minutos.)*

3. Usa un contrato comprobado con las rutas, pero sin criterios de aceptación. ¿Qué parte de la mejora viene de las rutas y qué parte de los criterios? Los resultados con las catorce tareas ya están en `resultados_ejercicios.json`; para verlo tú, basta con tres, por ejemplo T10, T17 y T19. *(Sin GPU con los resultados guardados; GPU, unos 30 minutos, con esas tareas.)*

4. Busca en el banco de tareas un contrato con un criterio que no se pueda comprobar con una prueba automática. Reescríbelo para que se pueda. *(Sin GPU.)*

5. El contrato comprobado rechaza al editar cualquier ruta que no esté permitida. ¿Rechaza también leerla? Mira el código de `HerramientasConContrato` y di qué herramientas vigila y cuáles no. *(Sin GPU.)*

6. Pide al agente que escriba él mismo el contrato antes de empezar, a partir de la petición y la lista de archivos, y úsalo como contrato comprobado. ¿Resuelve tantas tareas como con el contrato escrito por nosotros? Compara un contrato suyo con el nuestro. Los resultados con las catorce tareas ya están en `resultados_ejercicios.json`; para verlo tú, basta con tres, por ejemplo T10, T17 y T19. *(Sin GPU con los resultados guardados; GPU, unos 40 minutos, con esas tareas.)*

7. La puerta exige que la respuesta cite cada criterio por su número, pero no comprueba que se cumpla. Para la tarea del nuevo precio, escribe una función que compruebe de verdad los tres criterios sobre la copia del repositorio, importando `presupuestos`, `avisos` y leyendo las condiciones. *(Sin GPU.)*

8. ¿Qué pasa si el contrato está mal, por ejemplo si olvida una ruta necesaria? Quita `config.py` de las rutas permitidas de la tarea del nuevo precio y ejecútala con el contrato comprobado. ¿Qué hace el agente cuando le rechazan la edición? *(GPU, unos 15 minutos.)*

9. Calcula el coste por tarea resuelta de las tres variantes, en tokens de entrada y en segundos. ¿Compensa el contrato comprobado? *(Sin GPU.)*

10. **Proyecto.** Escribe un contrato para cada una de tres tareas reales de un repositorio tuyo, con objetivo, rutas y criterios de aceptación. Dáselos a un agente de programación, comercial o el del libro, y anota en qué tareas le ayudó el contrato y en cuáles le estorbó.

## Referencias

Modelo [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). La definición de terminado, en la [Guía de Scrum](https://scrumguides.org/scrum-guide.html). La tienda Ciclos Norte, su programa, sus tareas y sus contratos son ficticios. Código y experimentos propios.